# Matriz fundamental

Antes de empezar me gustaría mencionar algunos puntos a considerar. 

1. Se usa un dado justo de seis caras.

2. El juego termina cuando el jugador cae exactamente en la casilla $(20)$. Si el lanzamiento rebasa la casilla $(20)$, la ficha permanece en la misma casilla.

3. El ratón inicia en la casilla $(0)$, la casilla $(7)$ es comida y la casilla $(8)$ es shock.

4. En cada casilla no absorbente, el ratón elige uniformemente al azar entre las salidas disponibles. Las casillas $7$ y $8$ son absorbentes.


## Serpientes y escaleras

**Sol.**

Sea $X_n$ la casilla ocupada por el jugador después de la $n$-ésima tirada. Como el resultado de la siguiente tirada determina la nueva casilla únicamente a partir de la casilla actual, el juego se modela como una cadena de Markov a tiempo discreto.

La casilla $20$ es absorbente, porque al llegar a ella termina el juego. Las transiciones especiales del tablero son

$$
3\to 11,\qquad 15\to 19,\qquad 13\to 4,\qquad 17\to 9.
$$

Sea $T$ el número de tiradas necesarias para terminar el juego. Entonces

$$
T=\min\{n\geq 0:X_n=20\}.
$$

Para cada estado transitorio $i=0,1,\ldots,19$, se define

$$
e_i=E_i(T),
$$

donde $E_i(T)$ representa el número esperado de tiradas necesarias para llegar a la casilla $20$ cuando el jugador inicia en la casilla $i$. Además,

$$
e_{20}=0.
$$

Desde una casilla transitoria $i$ se realiza una tirada y después se continúa desde la casilla alcanzada. Para ello,

$$
e_i=1+\frac{1}{6}\sum_{d=1}^{6}e_{\varphi(i,d)},\qquad i=0,1,\ldots,19,
$$

donde $\varphi(i,d)$ es la casilla obtenida después de lanzar el valor $d$. Si $i+d>20$, entonces $\varphi(i,d)=i$, porque el lanzamiento rebasa la casilla final y la ficha permanece en la misma casilla. Si $i+d$ coincide con el inicio de una serpiente o escalera, se aplica la transición especial correspondiente.

En forma matricial, si $Q$ es la submatriz de transición entre los estados transitorios $0,1,\ldots,19$, entonces

$$
e=\mathbf{1}+Qe.
$$

Por tanto,

$$
(I-Q)e=\mathbf{1}.
$$

La matriz fundamental de la cadena absorbente es

$$
N=(I-Q)^{-1},
$$

y el vector de tiempos esperados se obtiene mediante

$$
e=N\mathbf{1}.
$$

La entrada $e_0$ corresponde al número promedio de tiradas necesarias cuando el jugador inicia antes de la casilla $1$.


In [1]:
import numpy as np
import sympy as sp

In [2]:
# Datos del tablero
casilla_final = 20
estados = list(range(casilla_final + 1))
estados_transitorios = list(range(casilla_final))

# Escaleras y serpientes
transiciones_especiales = {
    3: 11,
    15: 19,
    13: 4,
    17: 9
}

def mover(posicion, dado):
    nueva = posicion + dado

    if nueva > casilla_final:
        return posicion

    if nueva in transiciones_especiales:
        return transiciones_especiales[nueva]

    return nueva

In [3]:
# Matriz de transición
P = sp.zeros(casilla_final + 1, casilla_final + 1)

for i in estados:
    if i == casilla_final:
        P[i, i] = 1
    else:
        for dado in range(1, 7):
            j = mover(i, dado)
            P[i, j] += sp.Rational(1, 6)

# Submatriz de estados transitorios
Q = P[:casilla_final, :casilla_final]

# Matriz fundamental
I = sp.eye(casilla_final)
N = (I - Q).inv()

# Vector de tiempos esperados
unos = sp.ones(casilla_final, 1)
e = N * unos

promedio_exact = sp.simplify(e[0])
promedio_decimal = sp.N(promedio_exact, 15)

print("Número promedio exacto de tiradas desde la casilla 0:")
print(promedio_exact)
print()
print("Número promedio aproximado de tiradas desde la casilla 0:")
print(float(promedio_decimal))

Número promedio exacto de tiradas desde la casilla 0:
9627344849/820909548

Número promedio aproximado de tiradas desde la casilla 0:
11.727656076671677


Por tanto, analíticamente se obtiene

$$
E_0(T)=\frac{9627344849}{820909548}\approx 11.7276560767.
$$

Ent., el número promedio de tiradas necesarias para terminar el juego es aproximadamente

$$
11.728\text{ tiradas}.
$$


Para la simulación, en cada repetición se inicia en la casilla $0$, se lanzan dados justos hasta caer exactamente en la casilla $20$, y se registra el número de tiradas utilizadas. La media de esos valores aproxima el número esperado de tiradas.


In [4]:
def simular_juego(rng):
    posicion = 0
    tiradas = 0

    while posicion != casilla_final:
        dado = int(rng.integers(1, 7))
        posicion = mover(posicion, dado)
        tiradas += 1

    return tiradas

n_simulaciones = 200_000
rng = np.random.default_rng(20260513)

muestra_tiradas = np.array(
    [simular_juego(rng) for _ in range(n_simulaciones)],
    dtype=float
)

media_simulada = muestra_tiradas.mean()

print("Número de simulaciones:", n_simulaciones)
print("Media simulada de tiradas:", media_simulada)
print("Valor analítico:", float(promedio_decimal))

Número de simulaciones: 200000
Media simulada de tiradas: 11.718325
Valor analítico: 11.727656076671677


Podemos notar que la media simulada queda cerca del valor analítico. La diferencia se debe al error propio de la simulación de Monte Carlo.


## Problema del ratón

**Sol.**

La casilla $7$ representa comida y la casilla $8$ representa shock. Ambas casillas son absorbentes. La cantidad solicitada es

$$
P_0(\text{llegar a comida}).
$$

Se define

$$
u_i=P_i(\text{llegar a comida antes que a shock}),
$$

para cada estado $i$. Como la comida y el shock son absorbentes,

$$
u_7=1,\qquad u_8=0.
$$

De acuerdo con las salidas disponibles del tablero, las transiciones no absorbentes son

$$
0:\{1,2\},\qquad
1:\{0,3,7\},\qquad
2:\{0,3,8\},
$$

$$
3:\{1,2,4,5\},\qquad
4:\{3,6,7\},\qquad
5:\{3,6,8\},\qquad
6:\{4,5\}.
$$

Como el ratón elige uniformemente al azar entre sus salidas disponibles,entonces tenemos que:

$$
u_0=\frac{u_1+u_2}{2},
$$

$$
u_1=\frac{u_0+u_3+u_7}{3},\qquad
u_2=\frac{u_0+u_3+u_8}{3},
$$

$$
u_3=\frac{u_1+u_2+u_4+u_5}{4},
$$

$$
u_4=\frac{u_3+u_6+u_7}{3},\qquad
u_5=\frac{u_3+u_6+u_8}{3},
$$

$$
u_6=\frac{u_4+u_5}{2}.
$$

Sustituyendo $u_7=1$ y $u_8=0$, queda

$$
u_0=\frac{u_1+u_2}{2},
$$

$$
u_1=\frac{u_0+u_3+1}{3},\qquad
u_2=\frac{u_0+u_3}{3},
$$

$$
u_3=\frac{u_1+u_2+u_4+u_5}{4},
$$

$$
u_4=\frac{u_3+u_6+1}{3},\qquad
u_5=\frac{u_3+u_6}{3},
$$

$$
u_6=\frac{u_4+u_5}{2}.
$$

Este sistema también puede resolverse mediante matriz fundamental. Si la matriz de transición se escribe en forma absorbente como

$$
P=
\begin{pmatrix}
Q & R\\
0 & I
\end{pmatrix},
$$

entonces

$$
N=(I-Q)^{-1}
$$

y

$$
B=NR
$$

contiene las probabilidades de absorción. La entrada correspondiente al estado inicial $0$ y a la casilla absorbente $7$ da la probabilidad de llegar a la comida.


In [5]:
# Estados del problema del ratón
estados_transitorios_raton = [0, 1, 2, 3, 4, 5, 6]
estados_absorbentes_raton = [7, 8]

salidas = {
    0: [1, 2],
    1: [0, 3, 7],
    2: [0, 3, 8],
    3: [1, 2, 4, 5],
    4: [3, 6, 7],
    5: [3, 6, 8],
    6: [4, 5],
}

P_raton = sp.zeros(9, 9)

for i in range(9):
    if i in estados_absorbentes_raton:
        P_raton[i, i] = 1
    else:
        prob = sp.Rational(1, len(salidas[i]))
        for j in salidas[i]:
            P_raton[i, j] += prob

Q_raton = P_raton.extract(estados_transitorios_raton, estados_transitorios_raton)
R_raton = P_raton.extract(estados_transitorios_raton, estados_absorbentes_raton)

N_raton = (sp.eye(len(estados_transitorios_raton)) - Q_raton).inv()
B_raton = N_raton * R_raton

fila_estado_0 = estados_transitorios_raton.index(0)
columna_comida = estados_absorbentes_raton.index(7)

probabilidad_comida_exacta = sp.simplify(B_raton[fila_estado_0, columna_comida])
probabilidad_comida_decimal = sp.N(probabilidad_comida_exacta, 15)

print("Probabilidad exacta de alcanzar la comida desde la casilla 0:")
print(probabilidad_comida_exacta)
print()
print("Probabilidad aproximada:")
print(float(probabilidad_comida_decimal))

Probabilidad exacta de alcanzar la comida desde la casilla 0:
1/2

Probabilidad aproximada:
0.5


Por lo tanto,

$$
P_0(\text{llegar a comida})=\frac{1}{2}=0.5.
$$

Ent., la probabilidad de que el ratón, iniciando en la casilla $0$, alcance la comida es

$$
0.5.
$$


Para la simulación, en cada repetición el ratón inicia en la casilla $0$ y se mueve uniformemente entre las salidas disponibles hasta llegar a la comida o al shock. 


In [6]:
def simular_raton(rng):
    estado = 0

    while estado not in estados_absorbentes_raton:
        estado = int(rng.choice(salidas[estado]))

    return estado == 7

n_simulaciones_raton = 200_000
rng_raton = np.random.default_rng(20260514)

resultados_raton = np.array(
    [simular_raton(rng_raton) for _ in range(n_simulaciones_raton)],
    dtype=float
)

probabilidad_simulada = resultados_raton.mean()

print("Número de simulaciones:", n_simulaciones_raton)
print("Probabilidad simulada de alcanzar la comida:", probabilidad_simulada)
print("Valor analítico:", float(probabilidad_comida_decimal))

Número de simulaciones: 200000
Probabilidad simulada de alcanzar la comida: 0.500075
Valor analítico: 0.5


La probabilidad simulada queda cerca del valor analítico. La diferencia se debe al error propio de la simulación de Monte Carlo.

## Conclusión

El número promedio de tiradas necesarias para terminar el juego de serpientes y escaleras, iniciando en la casilla $0$, es

$$
E_0(T)\approx 11.7276560767.
$$

La probabilidad de que el ratón, iniciando en la casilla $0$, alcance la comida es

$$
P_0(\text{llegar a comida})=0.5.
$$
